# MS MARCO RARS-v8 Cutoff-Aware Frozen-Index Sidecar

## Goal

Run the first frozen V8 development experiment at implementation commit `c9d95f15d55e7700db069da69567157f2eed469e`. V8 does **not** train an encoder or query adapter. It learns one orthonormal rank-16 document-residual subspace from Top-10 promotion/protection margins, then evaluates five-fold out-of-fold int8 sidecar scores against both the frozen M32 base and a storage-matched residual-PCA sidecar.

This is outcome-informed development on `oracle_design`, not independent confirmation or an official MS MARCO result. `oracle_audit` and `future_method_holdout` must remain unopened.

## Frozen execution boundary

The run has three possible decisions:

- `GO_TO_RARS_ALGORITHM_CONFIRMATION_PROTOCOL`: RARS clears Base and PCA development gates.
- `GO_TO_GENERIC_SIDECAR_CONFIRMATION_PROTOCOL`: the compact sidecar signal clears the common gate, but RARS-over-PCA is not established.
- `STOP_V8_CUTOFF_SIDECAR`: stop this method line without post-hoc retuning.

Even a GO does not open the future role. It only permits a qrels-free full-corpus encoding stage; a new hash-bound evaluation protocol is still required afterward.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

EXPERIMENT_PYTHON = sys.executable
subprocess.run([
    EXPERIMENT_PYTHON, '-m', 'pip', 'install', '-q',
    'faiss-gpu-cu12==1.12.0', 'pytest>=8,<9',
], check=True)
NUMPY_TARGET = Path('/content/rars-v8-numpy126')
if NUMPY_TARGET.exists():
    shutil.rmtree(NUMPY_TARGET)
NUMPY_TARGET.mkdir(parents=True)
subprocess.run([
    EXPERIMENT_PYTHON, '-m', 'pip', 'install', '-q', '--no-deps',
    '--target', str(NUMPY_TARGET), 'numpy==1.26.4',
], check=True)
EXPERIMENT_ENV = os.environ.copy()
EXPERIMENT_ENV['PYTHONPATH'] = os.pathsep.join(filter(None, [
    str(NUMPY_TARGET), EXPERIMENT_ENV.get('PYTHONPATH', ''),
]))
EXPERIMENT_ENV['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
installed_numpy_version = subprocess.check_output([
    EXPERIMENT_PYTHON, '-c', 'import numpy; print(numpy.__version__)',
], text=True, env=EXPERIMENT_ENV).strip()
assert installed_numpy_version == '1.26.4', installed_numpy_version

from google.colab import drive
drive.mount('/content/drive')

import hashlib, json

TRAINING_COMMIT = 'bb9b106e69b9a453756fd800665f701614ce67b3'
V3_IMPLEMENTATION_COMMIT = '05c2ae43b7d11783460822d10c590240dab1a399'
V6_IMPLEMENTATION_COMMIT = '26a7717b964eed979b3bf7a3149d0d24e9bce3f1'
V8_IMPLEMENTATION_COMMIT = 'c9d95f15d55e7700db069da69567157f2eed469e'
REPO_URL = 'https://github.com/ravan-chuang/Embedding_Compression_for_RAG_Retrieval.git'
TRAIN_REPO = Path('/content/Embedding_Compression_for_RAG_Retrieval_rars_v2_2')
V3_REPO = Path('/content/Embedding_Compression_for_RAG_Retrieval_rars_v3')
V8_REPO = Path('/content/Embedding_Compression_for_RAG_Retrieval_rars_v8')
PARENT_WORK = Path('/content') / f'rars-v2.2-{TRAINING_COMMIT[:12]}'
V3_WORK = Path('/content') / f'rars-v3-{V3_IMPLEMENTATION_COMMIT[:12]}'
for work in (PARENT_WORK, V3_WORK):
    if work.exists():
        shutil.rmtree(work)
    work.mkdir(parents=True)
PARENT_BUNDLES = PARENT_WORK / 'bundles'
PARENT_CANDIDATE_CACHE = PARENT_WORK / 'candidate-cache'
V3_BUNDLES = V3_WORK / 'bundles'

DRIVE = Path('/content/drive/MyDrive/rag-pq-checkpoints')
CACHE = DRIVE / 'msmarco_basis_gate0_cache'
CLEAN = DRIVE / 'rars_clean_split_v1'
PCA = DRIVE / 'rars_pca_comparator_v1'
INDEX = DRIVE / 'msmarco_1m_pq_residual_gate3/frozen_ivfpq_m32_nlist512.index'
V6_PACKET = DRIVE / 'rars-v6-1m-headroom' / V6_IMPLEMENTATION_COMMIT[:12]
V8_ROOT = DRIVE / 'rars-v8-cutoff-sidecar' / V8_IMPLEMENTATION_COMMIT[:12]
DEVELOPMENT = V8_ROOT / 'development'
SIDECARS = V8_ROOT / 'sidecars'

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

def verify_record(path, record):
    path = Path(path)
    assert path.is_file(), path
    assert path.stat().st_size == int(record['bytes']), path
    assert sha256_file(path) == record['sha256'], path

def clone_exact(destination, commit):
    if destination.exists():
        shutil.rmtree(destination)
    subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(destination)], check=True)
    subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', commit], check=True)
    head = subprocess.check_output(['git', '-C', str(destination), 'rev-parse', 'HEAD'], text=True).strip()
    dirty = subprocess.check_output(['git', '-C', str(destination), 'status', '--porcelain'], text=True).strip()
    assert head == commit, (head, commit)
    assert not dirty, dirty

print('Fresh experiment-subprocess NumPy:', installed_numpy_version)

In [ ]:
clone_exact(TRAIN_REPO, TRAINING_COMMIT)
clone_exact(V3_REPO, V3_IMPLEMENTATION_COMMIT)
clone_exact(V8_REPO, V8_IMPLEMENTATION_COMMIT)

V3_PROTOCOL_PATH = V3_REPO / 'protocols/rars_v3_oracle_first_feasibility_v1.json'
V8_PROTOCOL_PATH = V8_REPO / 'protocols/rars_v8_cutoff_sidecar_v1.json'
protocol = json.loads(V8_PROTOCOL_PATH.read_text())
assert protocol['status'] == 'FROZEN_BEFORE_FIRST_V8_DEVELOPMENT_RUN'
assert protocol['parent_lineage']['v6_source_commit'] == V6_IMPLEMENTATION_COMMIT
assert protocol['method']['basis_constraints']['single_symmetric_basis'] is True
assert protocol['method']['coefficient_dtype'] == 'int8'
assert protocol['data_policy']['cross_validation']['out_of_fold_metrics_only'] is True
assert protocol['development_gate']['future_access_authorized_by_development_result'] is False

subprocess.run([
    EXPERIMENT_PYTHON, '-m', 'pytest', '-q',
    'tests/test_rars_v2_2_core.py',
    'tests/test_freeze_rars_v2_2_inner_bundles.py',
    'tests/test_build_msmarco_rars_v2_boundary_bundles.py',
], cwd=TRAIN_REPO, check=True, env=EXPERIMENT_ENV)
subprocess.run([
    EXPERIMENT_PYTHON, '-m', 'pytest', '-q',
    'tests/test_rars_v3_oracle_core.py',
    'tests/test_build_msmarco_rars_v3_oracle_bundles.py',
    'tests/test_materialize_rars_v3_role_labels.py',
], cwd=V3_REPO, check=True, env=EXPERIMENT_ENV)
subprocess.run([
    EXPERIMENT_PYTHON, '-m', 'pytest', '-q',
    'tests/test_rars_v8_cutoff_sidecar_core.py',
    'tests/test_rars_v8_cutoff_sidecar_protocol_contract.py',
    'tests/test_train_rars_v8_cutoff_sidecar_contract.py',
    'tests/test_build_rars_v8_frozen_sidecars_contract.py',
    'tests/test_verify_rars_v6_1m_headroom_packet.py',
], cwd=V8_REPO, check=True, env=EXPERIMENT_ENV)
print('Exact V8 implementation commit:', V8_IMPLEMENTATION_COMMIT)
print('Frozen V8 protocol SHA-256:', sha256_file(V8_PROTOCOL_PATH))

In [ ]:
required = [
    CACHE / 'embeddings.fp16.memmap',
    CACHE / 'doc_ids.int64.memmap',
    CACHE / 'query_vectors.fp32.npy',
    CACHE / 'qrels_subset.json',
    INDEX,
    PCA / 'bases/pca_unweighted_rank16.float32.npy',
    PCA / 'sidecars/scales_pca_rank16.float32.npy',
    PCA / 'sidecars/codes_pca_rank16.int8.memmap',
    CLEAN / 'selected_config.json',
    CLEAN / 'bases/score_error_weighted_rank16.npy',
    CLEAN / 'sidecars/scales_score_error_weighted_rank16.float32.npy',
    CLEAN / 'sidecars/codes_score_error_weighted_rank16.int8.memmap',
    V6_PACKET / 'headroom_result.json',
    V6_PACKET / 'headroom_complete.json',
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, {'missing_artifacts': missing}
assert shutil.disk_usage('/content').free >= 8_000_000_000, 'Need 8 GB local disk'
assert not DEVELOPMENT.exists() or not any(DEVELOPMENT.iterdir()), (
    'The durable V8 development packet is non-empty. Do not overwrite it.'
)
v6_verification = subprocess.check_output([
    EXPERIMENT_PYTHON, str(V8_REPO / 'scripts/verify_rars_v6_1m_headroom_packet.py'),
    '--packet-root', str(V6_PACKET),
], text=True, cwd=V8_REPO, env=EXPERIMENT_ENV)
v6_summary = json.loads(v6_verification)
assert v6_summary['status'] == 'RARS_V6_1M_HEADROOM_PACKET_VERIFIED'
assert v6_summary['formal_decision'] == 'GO_TO_V6_LOSS_IMPLEMENTATION'
assert v6_summary['source_commit'] == V6_IMPLEMENTATION_COMMIT
print(json.dumps(v6_summary, indent=2, allow_nan=False))

## Rebuild the exact development bundle

The next cells rematerialize the pinned v2.2 parent bundle, then the qrels-free V3 role split. Only `oracle_design` labels are copied from the already frozen parent arrays. No qrels parser is used by the V3 materializer; `oracle_audit` remains unlabeled and the 803-query future role remains identity-only.

In [ ]:
builder = [
    EXPERIMENT_PYTHON, str(TRAIN_REPO / 'scripts/build_msmarco_rars_v2_boundary_bundles.py'),
    '--inner-only',
    '--embeddings', str(CACHE / 'embeddings.fp16.memmap'),
    '--doc-ids', str(CACHE / 'doc_ids.int64.memmap'),
    '--query-vectors', str(CACHE / 'query_vectors.fp32.npy'),
    '--index', str(INDEX),
    '--qrels', str(CACHE / 'qrels_subset.json'),
    '--train-split', str(TRAIN_REPO / 'splits/msmarco_rars_train_split.json'),
    '--validation-split', str(TRAIN_REPO / 'splits/msmarco_rars_validation_split.json'),
    '--cache-root', str(PARENT_CANDIDATE_CACHE),
    '--pca-config', str(TRAIN_REPO / 'results/rars_pca_comparator/selected_pca_config.json'),
    '--pca-basis', str(PCA / 'bases/pca_unweighted_rank16.float32.npy'),
    '--pca-scales', str(PCA / 'sidecars/scales_pca_rank16.float32.npy'),
    '--pca-codes', str(PCA / 'sidecars/codes_pca_rank16.int8.memmap'),
    '--rars-config', str(CLEAN / 'selected_config.json'),
    '--rars-basis', str(CLEAN / 'bases/score_error_weighted_rank16.npy'),
    '--rars-scales', str(CLEAN / 'sidecars/scales_score_error_weighted_rank16.float32.npy'),
    '--rars-codes', str(CLEAN / 'sidecars/codes_score_error_weighted_rank16.int8.memmap'),
    '--output-root', str(PARENT_BUNDLES),
    '--residual-batch-size', '20000',
]
subprocess.run(builder, check=True, cwd=TRAIN_REPO, env=EXPERIMENT_ENV)
subprocess.run([
    EXPERIMENT_PYTHON, str(TRAIN_REPO / 'scripts/freeze_rars_v2_2_inner_bundles.py'),
    '--bundle-root', str(PARENT_BUNDLES),
    '--query-vectors', str(CACHE / 'query_vectors.fp32.npy'),
    '--train-split', str(TRAIN_REPO / 'splits/msmarco_rars_train_split.json'),
    '--outer-validation-split', str(TRAIN_REPO / 'splits/msmarco_rars_validation_split.json'),
    '--clean-test-split', str(TRAIN_REPO / 'splits/msmarco_rars_test_split.json'),
    '--source-commit', TRAINING_COMMIT,
], check=True, cwd=TRAIN_REPO, env=EXPERIMENT_ENV)
print('Exact v2.2 parent rematerialized.')

In [ ]:
subprocess.run([
    EXPERIMENT_PYTHON, str(V3_REPO / 'scripts/build_msmarco_rars_v3_oracle_bundles.py'),
    '--parent-inner-train-bundle', str(PARENT_BUNDLES / 'inner_train'),
    '--doc-ids', str(CACHE / 'doc_ids.int64.memmap'),
    '--output-root', str(V3_BUNDLES),
    '--protocol', str(V3_PROTOCOL_PATH),
    '--source-commit', V3_IMPLEMENTATION_COMMIT,
    '--n-docs', '1000000',
], check=True, cwd=V3_REPO, env=EXPERIMENT_ENV)
candidate_summary = json.loads((V3_BUNDLES / 'v3_oracle_bundle_freeze_summary.json').read_text())
assert candidate_summary['status'] == 'V3_QRELS_FREE_CANDIDATE_BUNDLES_FROZEN'
assert candidate_summary['qrels_opened_or_parsed'] is False
ROLE_LABEL_FILES = {
    'candidate_relevance.uint8.npy', 'relevant_counts.int32.npy',
    'v3_role_labels_started.json', 'v3_role_labels_manifest.json',
}
for role in ('oracle_design', 'oracle_audit'):
    assert not ROLE_LABEL_FILES.intersection(path.name for path in (V3_BUNDLES / role).iterdir())
future_files = {path.name for path in (V3_BUNDLES / 'future_method_holdout').iterdir()}
assert future_files == {'query_manifest.json', 'v3_identity_manifest.json'}
print('Qrels-free V3 role identities and candidates are frozen.')

In [ ]:
subprocess.run([
    EXPERIMENT_PYTHON, str(V3_REPO / 'scripts/materialize_rars_v3_role_labels.py'),
    '--bundle-root', str(V3_BUNDLES),
    '--parent-inner-train-bundle', str(PARENT_BUNDLES / 'inner_train'),
    '--role', 'oracle_design',
    '--source-commit', V3_IMPLEMENTATION_COMMIT,
    '--protocol', str(V3_PROTOCOL_PATH),
], check=True, cwd=V3_REPO, env=EXPERIMENT_ENV)
design_labels = json.loads(
    (V3_BUNDLES / 'oracle_design' / 'v3_role_labels_manifest.json').read_text()
)
assert design_labels['status'] == 'ROLE_LABELS_MATERIALIZED_FROM_FROZEN_PARENT'
assert design_labels['role_id'] == 'oracle_design'
assert design_labels['label_source']['qrels_opened_or_parsed'] is False
assert not ROLE_LABEL_FILES.intersection(
    path.name for path in (V3_BUNDLES / 'oracle_audit').iterdir()
)
assert {path.name for path in (V3_BUNDLES / 'future_method_holdout').iterdir()} == future_files
print('Only oracle_design labels materialized; audit unlabeled; future identity-only.')

## Run five-fold V8 development

The trainer verifies the V6 packet again, checks every V3 input hash, mines static cutoff pairs, fits five out-of-fold RARS bases, and scores both PCA and RARS through int8 codes. Hyperparameters are fixed; no epoch or seed is selected from outcomes. Do not interrupt the cell or edit the durable output directory.

In [ ]:
DEVELOPMENT.parent.mkdir(parents=True, exist_ok=True)
subprocess.run([
    EXPERIMENT_PYTHON, str(V8_REPO / 'scripts/train_rars_v8_cutoff_sidecar.py'),
    '--design-role-dir', str(V3_BUNDLES / 'oracle_design'),
    '--v6-packet-root', str(V6_PACKET),
    '--output-dir', str(DEVELOPMENT),
    '--protocol', str(V8_PROTOCOL_PATH),
    '--source-commit', V8_IMPLEMENTATION_COMMIT,
], check=True, cwd=V8_REPO, env=EXPERIMENT_ENV)
print('RARS-v8 development completed.')

In [ ]:
complete_path = DEVELOPMENT / 'development_complete.json'
result_path = DEVELOPMENT / 'development_result.json'
freeze_path = DEVELOPMENT / 'method_freeze.json'
complete = json.loads(complete_path.read_text())
result = json.loads(result_path.read_text())
freeze = json.loads(freeze_path.read_text())
allowed = {
    'GO_TO_RARS_ALGORITHM_CONFIRMATION_PROTOCOL',
    'GO_TO_GENERIC_SIDECAR_CONFIRMATION_PROTOCOL',
    'STOP_V8_CUTOFF_SIDECAR',
}
assert complete['status'] == 'RARS_V8_DEVELOPMENT_COMPLETE'
assert result['status'] == 'RARS_V8_DEVELOPMENT_COMPLETE'
assert freeze['status'] == 'RARS_V8_METHOD_FROZEN_AFTER_DEVELOPMENT'
assert complete['source_commit'] == V8_IMPLEMENTATION_COMMIT
assert complete['formal_decision'] == result['formal_decision'] == freeze['formal_decision']
assert result['formal_decision'] in allowed
assert complete['future_method_holdout_opened'] is False
assert complete['oracle_audit_opened'] is False
assert complete['full_corpus_sidecar_encoded'] is False
assert freeze['future_access_authorized'] is False
verify_record(DEVELOPMENT / 'development_started.json', complete['started'])
for filename, record in complete['outputs'].items():
    verify_record(DEVELOPMENT / filename, record)
missing_outputs = [
    name for name in protocol['required_development_outputs']
    if not (DEVELOPMENT / name).is_file()
]
assert not missing_outputs, missing_outputs
report = {
    'formal_decision': result['formal_decision'],
    'metrics': result['metrics'],
    'comparisons': result['comparisons'],
    'candidate_gap_recovery_fraction': result['candidate_gap_recovery_fraction'],
    'pair_support': result['pair_support'],
    'failed_gates': result['decision']['failed_gates'],
    'development_result_sha256': sha256_file(result_path),
    'method_freeze_sha256': sha256_file(freeze_path),
}
print(json.dumps(report, indent=2, allow_nan=False))

## Conditional qrels-free full-corpus encoding

This cell runs only when the closed development packet contains a GO decision. The builder accepts no qrels, query, audit, or future-role argument. It calibrates int8 scales over all one million residuals, writes row-aligned PCA and RARS sidecars, and checks the frozen index hash before and after. A STOP result skips this stage.

In [ ]:
go_decisions = {
    'GO_TO_RARS_ALGORITHM_CONFIRMATION_PROTOCOL',
    'GO_TO_GENERIC_SIDECAR_CONFIRMATION_PROTOCOL',
}
if result['formal_decision'] in go_decisions:
    assert not SIDECARS.exists() or not any(SIDECARS.iterdir()), (
        'The durable V8 sidecar output is non-empty. Do not overwrite it.'
    )
    subprocess.run([
        EXPERIMENT_PYTHON, str(V8_REPO / 'scripts/build_rars_v8_frozen_sidecars.py'),
        '--development-packet', str(DEVELOPMENT),
        '--embeddings', str(CACHE / 'embeddings.fp16.memmap'),
        '--doc-ids', str(CACHE / 'doc_ids.int64.memmap'),
        '--index', str(INDEX),
        '--output-root', str(SIDECARS),
        '--protocol', str(V8_PROTOCOL_PATH),
        '--source-commit', V8_IMPLEMENTATION_COMMIT,
        '--batch-size', '8192',
    ], check=True, cwd=V8_REPO, env=EXPERIMENT_ENV)
    sidecar_complete = json.loads((SIDECARS / 'sidecars_complete.json').read_text())
    assert sidecar_complete['status'] == 'RARS_V8_FULL_CORPUS_SIDECARS_COMPLETE'
    assert sidecar_complete['formal_decision'] == result['formal_decision']
    assert sidecar_complete['index_before'] == sidecar_complete['index_after']
    assert sidecar_complete['qrels_opened'] is False
    assert sidecar_complete['future_method_holdout_opened'] is False
    print(json.dumps(sidecar_complete, indent=2, allow_nan=False))
else:
    print('STOP decision: full-corpus encoding correctly skipped.')

## Return for audit

Return the printed development report plus `development_result.json`, `method_freeze.json`, and `development_complete.json`. If full-corpus encoding ran, also return `sidecars_result.json` and `sidecars_complete.json`.

Do not tune V8 after seeing this run. A GO still requires a separately authored and committed confirmation protocol that binds these exact hashes before any future outcome is materialized. A generic-sidecar GO must not be reported as RARS superiority; a STOP closes V8 without rescue.